In [28]:
import duckdb
import polars as pl
import pandas as pd

In [29]:
con = duckdb.connect("../data/m5.db")

In [30]:
!ls ../data/processed/*

../data/processed/level_01_daily_total.parquet
../data/processed/level_02_daily_state.parquet
../data/processed/level_03_daily_cat.parquet
../data/processed/level_04_daily_dept.parquet
../data/processed/level_05_daily_state_cat.parquet
../data/processed/level_06_daily_store.parquet
../data/processed/level_07_daily_state_dept.parquet
../data/processed/level_08_daily_store_cat.parquet
../data/processed/level_09_daily_store_dept.parquet
../data/processed/level_10_daily_item.parquet
../data/processed/level_10_weekly_item.parquet
../data/processed/level_11_daily_item_state.parquet
../data/processed/level_11_weekly_item_state.parquet
../data/processed/level_12_daily_item_store.parquet
../data/processed/level_12_weekly_item_store.parquet


In [36]:
DATASET_PATH = "../data/processed/level_11_daily_item_state.parquet"

In [37]:
duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{DATASET_PATH}')
    LIMIT 5
""").df()

,agg_id,item_id,dept_id,cat_id,store_id,state_id,date,sales,gross_sales,avg_sell_price,...,dayofweek,weekofyear,is_weekend,cum7,cum14,cum21,cum28,cum35,cum42,cum365
0,FOODS_1_FOODS_FOODS_1_001_CA,FOODS_1_001,FOODS_1,FOODS,TOTAL,CA,2011-01-29,6.0,12.0,2.0,...,5,4,1,31.0,68.0,105.0,157.0,194.0,236.0,1473.0
1,FOODS_1_FOODS_FOODS_1_001_CA,FOODS_1_001,FOODS_1,FOODS,TOTAL,CA,2011-01-30,3.0,6.0,2.0,...,6,4,1,33.0,73.0,108.0,162.0,204.0,241.0,1475.0
2,FOODS_1_FOODS_FOODS_1_001_CA,FOODS_1_001,FOODS_1,FOODS,TOTAL,CA,2011-01-31,2.0,4.0,2.0,...,0,5,0,33.0,77.0,110.0,161.0,210.0,240.0,1477.0
3,FOODS_1_FOODS_FOODS_1_001_CA,FOODS_1_001,FOODS_1,FOODS,TOTAL,CA,2011-02-01,3.0,6.0,2.0,...,1,5,0,32.0,79.0,111.0,162.0,211.0,238.0,1479.0
4,FOODS_1_FOODS_FOODS_1_001_CA,FOODS_1_001,FOODS_1,FOODS,TOTAL,CA,2011-02-02,7.0,14.0,2.0,...,2,5,0,26.0,80.0,110.0,158.0,208.0,235.0,1472.0


In [43]:
df = duckdb.sql(f"""
    SELECT
    state_id, 
    cat_id,
    COUNT(DISTINCT(store_id)) AS stores,
    COUNT(DISTINCT(item_id)) AS items,
    SUM(sales) AS sales ,
    SUM(gross_sales) / 1000000 AS gross_sales_M ,
    COUNT(*) AS rows
    FROM read_parquet('{DATASET_PATH}')
    GROUP BY state_id, cat_id
    ORDER BY gross_sales_M DESC
""").df()

df.to_clipboard(index=False)
df.round(2)

,state_id,cat_id,stores,items,sales,gross_sales_M,rows
0,CA,FOODS,1,1437,19535863.0,48.32,2789217
1,TX,FOODS,1,1437,13172106.0,31.66,2789217
2,WI,FOODS,1,1437,13214458.0,31.16,2789217
3,CA,HOUSEHOLD,1,1047,6565267.0,26.84,2032227
4,TX,HOUSEHOLD,1,1047,4432169.0,16.42,2032227
5,WI,HOUSEHOLD,1,1047,3766654.0,13.85,2032227
6,CA,HOBBIES,1,565,3095587.0,10.80,1096665
7,TX,HOBBIES,1,565,1624130.0,7.04,1096665
8,WI,HOBBIES,1,565,1520939.0,5.48,1096665
